## Setup

In [225]:
import sys
sys.path.insert(0, 'E:\Code\Music')

In [226]:
from pathlib import Path
from PDMX.reading.music import MusicRender
import random
from collections import Counter
import torch.nn as nn
import torch
import pretty_midi
from torch.utils.data import TensorDataset, DataLoader
import scipy.io.wavfile as wav
import IPython.display as ipd
from tqdm import tqdm
import numpy as np

In [227]:
PDMX_dir = Path('E:\Code\music\dataset')
JSON_ROOT = PDMX_dir / 'data'
JSON_FILES = [str(file) for file in JSON_ROOT.rglob('*.json')]

GRID_16TH = np.linspace(0.25, 4.0, 16)
GRID_TRIP = np.linspace(1/3, 4.0, 12)
GRID = np.union1d(GRID_16TH, GRID_TRIP)

random.seed(42)

## Data Processing

In [228]:
def quantize(duration, resolution):
    note_value = duration / resolution
    quantized = min(GRID, key=lambda g: abs(g - note_value))
    return round(quantized * resolution, 1)

In [229]:
def extract_note_sequences(json_file: str) -> list[list]:
    note_sequences = []
    music = MusicRender()
    music.load(json_file)
    resolution = music.resolution
    
    for track in music.tracks:
        notes = [
            (note.pitch, quantize(note.duration, resolution), note.velocity)
            for note in track.notes
        ]
        
        note_sequences.append(notes)
    
    return note_sequences
    

In [230]:
# collecting sequences from JSONs
def get_seqs_from_files(json_files, k=5000):
    all_sequences = []
    for path in (random.sample(json_files, k)):
        notes = extract_note_sequences(path)
        all_sequences.extend(seq for seq in notes)
    return all_sequences

In [231]:
def make_mappings(vocab):
    idx_to_note = {idx : note for idx, note in enumerate(vocab)}
    note_to_idx = {note : idx for idx, note in idx_to_note.items()}
    return idx_to_note, note_to_idx

In [232]:
# i: 0 = pitch, 1 = beat, 2 = velocity
def create_vocab(sequence_list, i):
    all_vals = [note[i] for seq in sequence_list for note in seq]
    all_counts = Counter(all_vals)
    
    vocab = sorted(all_counts.keys())
    idx_to_note, note_to_idx = make_mappings(vocab)
    
    return vocab, idx_to_note, note_to_idx

In [233]:
def train_test_split(encoded_seqs, seq_length=64):
    x_data = []
    y_data = []
    
    for sequence in encoded_seqs:
        for i in range(len(sequence) - seq_length):
            x_data.append(sequence[i:i+seq_length])
            y_data.append(sequence[i+1:i+1+seq_length])    
    
    return x_data, y_data

In [234]:
all_sequences = get_seqs_from_files(JSON_FILES)

In [235]:
pitch_vocab, idx_to_note, note_to_idx = create_vocab(all_sequences, 0)
beat_vocab, idx_to_beat, beat_to_idx = create_vocab(all_sequences, 1)
vel_vocab, idx_to_vel, vel_to_idx = create_vocab(all_sequences, 2)

PITCH_VOCAB_SIZE, BEAT_VOCAB_SIZE, VEL_VOCAB_SIZE = [len(V) for V in (pitch_vocab, beat_vocab, vel_vocab)]

In [236]:
encoded_seqs = [
    [
        (note_to_idx[pitch], beat_to_idx[beat], vel_to_idx[vel]) 
        for pitch, beat, vel in sequence
    ] 
    for sequence in all_sequences
]

In [237]:
x_data, y_data = train_test_split(encoded_seqs)

## Model Architecture

In [238]:
class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, beat_vocab_size, vel_vocab_size, embed_dim=256, max_seq_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.beat_embedding = nn.Embedding(beat_vocab_size, embed_dim)
        self.vel_embedding = nn.Embedding(vel_vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=512, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        
        self.out = nn.Linear(embed_dim, vocab_size)
        self.beat_out = nn.Linear(embed_dim, beat_vocab_size)
        self.vel_out = nn.Linear(embed_dim, vel_vocab_size)
        
    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x[:, :, 0]) + self.beat_embedding(x[:, :, 1]) + self.vel_embedding(x[:, :, 2]) + self.pos_embedding(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1), device=x.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.out(x), self.beat_out(x), self.vel_out(x)
    
    def generate(self, seed, temp=0.8):    
        was_training = self.training
        self.eval()
        
        seq_length = seed.shape[1]
        new_seq = seed.detach().clone()
        with torch.no_grad():
            for _ in range(seq_length):
                pitch_out, beat_out, vel_out = self(new_seq)
                
                pitch_logits = torch.softmax(pitch_out[0, -1, :] / temp, -1)
                beat_logits = torch.softmax(beat_out[0, -1, :] / temp, -1)
                vel_logits = torch.softmax(vel_out[0, -1, :] / temp, -1)
                
                sample_pitch = torch.multinomial(pitch_logits, 1).view(1, 1, 1)
                sample_beat = torch.multinomial(beat_logits, 1).view(1, 1, 1)
                sample_vel = torch.multinomial(vel_logits, 1).view(1, 1, 1)

                combined_sample = torch.cat([sample_pitch, sample_beat, sample_vel], dim=2)
                
                new_seq = torch.cat([new_seq, combined_sample], dim=1)[:, 1:]
        
        self.train(was_training)
        return new_seq

### Device, DataLoader, Loss, Optimizer & Scheduler

- **Device**: moves all computation to GPU if available; otherwise CPU.
- **CrossEntropyLoss**: fused log-softmax + NLL for stable 95-class prediction at every position.
- **AdamW** (`lr=3e-4`, `weight_decay=1e-2`): adaptive per-parameter LR with correctly decoupled weight decay.
- **CosineAnnealingLR**: smoothly decays LR from `3e-4` to ~0 over `T_max` epochs so the model settles precisely near a minimum.

In [239]:
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [240]:
def make_dataset_and_loader(x, y, batch_size=64):
    x_tensor = torch.tensor(x, dtype=torch.long)
    y_tensor = torch.tensor(y, dtype=torch.long)

    dataset = TensorDataset(x_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size, shuffle=True)
    
    return dataset, loader

In [241]:
dataset, loader = make_dataset_and_loader(x_data, y_data)

print(f'Dataset size : {len(dataset):,} sequences\n'
      f'Batches/epoch: {len(loader):,}')

Dataset size : 2,019,739 sequences
Batches/epoch: 31,559


In [242]:
device = get_device()
model = MusicTransformer(PITCH_VOCAB_SIZE, BEAT_VOCAB_SIZE, VEL_VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(f'Training on: {device}')

Training on: cuda


## Training Loop

Each step:
1. Zero gradients — clear accumulated gradients from the previous step.
2. Forward pass — model produces `(batch, 64, vocab_size)` logits.
3. Reshape — flatten to `(batch×64, vocab_size)` and `(batch×64,)` for CrossEntropyLoss.
4. Backward — compute gradients via autograd.
5. Clip — rescale the global gradient norm to ≤ 1.0 to prevent exploding gradients.
6. Step — update weights; then step the LR scheduler once per epoch.

In [243]:
def train_model(model, loader, criterion, optimizer, scheduler, device, epochs=10):    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for x_batch, y_batch in tqdm(loader):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            pitch_logits, beat_logits, vel_logits = model(x_batch)                         # (batch, 64, vocab_size)
            
            pitch_loss = criterion(pitch_logits.view(-1, PITCH_VOCAB_SIZE), y_batch[:, :, 0].view(-1))
            beat_loss = criterion(beat_logits.view(-1, BEAT_VOCAB_SIZE), y_batch[:, :, 1].view(-1))
            vel_loss = criterion(vel_logits.view(-1, VEL_VOCAB_SIZE), y_batch[:, :, 2].view(-1))
            
            loss = pitch_loss + beat_loss + vel_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(loader)
        print(f'Epoch {epoch+1:>2}/{epochs}  loss: {avg_loss:.4f}  lr: {scheduler.get_last_lr()[0]:.2e}')

In [244]:
train_model(model, loader, criterion, optimizer, scheduler, device)

100%|██████████| 31559/31559 [19:44<00:00, 26.65it/s]


Epoch  1/10  loss: 2.3251  lr: 2.93e-04


100%|██████████| 31559/31559 [19:43<00:00, 26.67it/s]


Epoch  2/10  loss: 1.8908  lr: 2.71e-04


100%|██████████| 31559/31559 [19:43<00:00, 26.66it/s]


...

Epoch  8/10  loss: 1.5298  lr: 2.86e-05


100%|██████████| 31559/31559 [19:44<00:00, 26.65it/s]


Epoch  9/10  loss: 1.5068  lr: 7.34e-06


100%|██████████| 31559/31559 [19:45<00:00, 26.61it/s]

Epoch 10/10  loss: 1.4940  lr: 0.00e+00


## Save Model

In [251]:
# saving the model
torch.save({
    'model_state_dict': model.state_dict(),
    'pitch_vocab_size': PITCH_VOCAB_SIZE,
    'beat_vocab_size': BEAT_VOCAB_SIZE,
    'vel_vocab_size': VEL_VOCAB_SIZE,
    'pitch_vocab': pitch_vocab,
    'beat_vocab': beat_vocab,
    'vel_vocab': vel_vocab,
    }, 'music_transformer.pth')

## Load Model

In [246]:
# model = MusicTransformer(110, BEAT_VOCAB_SIZE, 77)
# model.load_state_dict(torch.load('music_transformer.pth'))
# model.to(device)
# model.eval()

## Generating New Music

In [247]:
#take a given note sequence [(pitch, duration, velocity), ...] and return/output it as a midi
def notes_to_midi(note_sequence, output_path='output.mid', tempo=120, resolution=480.0):
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    instrument = pretty_midi.Instrument(program=0)
    
    seconds_per_beat = 60.0 / tempo
    elapsed_time = 0.0
    for pitch, duration_ticks, velocity in note_sequence:
        duration_seconds = duration_ticks / resolution * seconds_per_beat
        note = pretty_midi.Note(velocity=velocity, pitch=pitch, start=elapsed_time, end=elapsed_time + duration_seconds)
        instrument.notes.append(note)
        elapsed_time += duration_seconds
    
    pm.instruments.append(instrument)
    pm.write(output_path)
    return pm

In [269]:
# take a 3D Matrix representing a single sequence and output it to [(pitch, duration, velocity),...]
def format_notes(sequence):    
    new_seq = sequence.flatten()
    idx_list = new_seq.tolist()

    note_sequence = list(
        zip(
            [idx_to_note[i] for i in idx_list[::3]],
            [idx_to_beat[j] for j in idx_list[1::3]],
            [idx_to_vel[k] for k in idx_list[2::3]],
        )
    )
    
    return note_sequence


In [270]:
# select a random sequence from the training data shape(1, 64, 3) - final dim is (pitch, duration, velocity)
seed_idx = torch.randint(0, len(x_data), (1,)).item()
seed = torch.tensor(x_data[seed_idx]).unsqueeze(0).to(device)


new_notes = model.generate(seed, temp=1.0)
note_sequence = format_notes(new_notes)
print(note_sequence)
pm = notes_to_midi(note_sequence)

sf2_path = r'c:\Users\James\Music\Samples\GeneralUser_GS_v2.0.3--doc_r6\GeneralUser-GS\GeneralUser-GS.sf2'
audio = pm.fluidsynth(fs=44100, sf2_path=sf2_path)
audio_int16 = (audio * 32767).astype('int16')
wav.write('output.wav', rate=44100, data=audio_int16)

[(67, 480.0, 50), (67, 240.0, 50), (67, 480.0, 50), (66, 240.0, 50), (66, 480.0, 50), (62, 240.0, 50), (62, 360.0, 50), (60, 360.0, 50), (59, 120.0, 50), (57, 120.0, 50), (59, 480.0, 50), (67, 240.0, 50), (67, 480.0, 50), (69, 240.0, 50), (69, 480.0, 50), (67, 240.0, 50), (67, 480.0, 50), (64, 240.0, 50), (64, 480.0, 50), (66, 120.0, 50), (67, 120.0, 50), (69, 240.0, 50), (69, 480.0, 50), (65, 240.0, 50), (62, 480.0, 50), (64, 240.0, 50), (65, 480.0, 50), (67, 240.0, 50), (67, 480.0, 50), (66, 240.0, 50), (66, 480.0, 50), (62, 240.0, 50), (62, 360.0, 50), (60, 360.0, 50), (59, 240.0, 50), (67, 240.0, 50), (67, 480.0, 50), (67, 240.0, 50), (67, 480.0, 50), (66, 240.0, 50), (66, 480.0, 50), (62, 240.0, 50), (62, 360.0, 50), (60, 360.0, 50), (59, 240.0, 50), (67, 120.0, 50), (69, 360.0, 50), (69, 240.0, 50), (69, 240.0, 50), (71, 240.0, 50), (71, 240.0, 50), (67, 240.0, 50), (66, 120.0, 50), (67, 360.0, 50), (62, 240.0, 50), (62, 240.0, 50), (67, 240.0, 50), (67, 240.0, 50), (69, 240.0, 5

In [271]:
ipd.Audio('output.wav', embed=True)